Forecasting Appointment Volume for Nexora Care Flow:

We can forcast the volume of clinic(s) appointment from the Nexora Care dataset. 

This projects to solves the volatile appointment demand of participating clinics thereby offering insights to administrators to have forward visibility into appointments, Staffing and room allocation. Because these clinics run tightly staffed, appointment-driven schedules rather than 24/7 shifts, even modest swings in daily demand have an outsized effect on wait times and staff workload. Administrators repeatedly report that they only learn a day is unusually busy once it is already underway

In [13]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print('Libraries imported successfully')

Libraries imported successfully


### Data Understanding and cleaning

In [ ]:
# Load the appointment dataset
path = r'C:\Users\Joshua\amdira-projects\Appointment-volume-forecasting\data\AppointmentRecords.csv'
df = pd.read_csv(path)
print('Dataset loaded successfully')
print(f'Dataset has {df.shape[0]} rows and {df.shape[1]} columns')

print('Preview of the dataset:')
print(df.head().to_string())

Dataset loaded successfully
Dataset has 129353 rows and 15 columns
Preview of the dataset:
   AppointmentID  ClinicID                      ClinicName          ClinicType  PatientID  ProviderID AppointmentDateTime AppointmentType     Status BookingChannel  BookingLeadTimeDays  IsHoliday     HolidayName  SeasonFlag  ChronicConditionFlag
0         529385         2          Nexora Care - Lakeside     Multi-Specialty     103642         202    2024-01-01 08:00     New Patient  Completed         Online                    2       True  New Year's Day  Flu Season                 False
1         500000         1         Nexora Care - Riverside        Primary Care     101551         101    2024-01-01 08:30     New Patient    No-Show         Online                   18       True  New Year's Day  Flu Season                 False
2         500011         1         Nexora Care - Riverside        Primary Care     100260         104    2024-01-01 08:30       Follow-Up  Completed       Referral        

In [21]:
print('Dataset information:')
print(df.info())

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129353 entries, 0 to 129352
Data columns (total 15 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   AppointmentID         129353 non-null  int64 
 1   ClinicID              129353 non-null  int64 
 2   ClinicName            129353 non-null  object
 3   ClinicType            129353 non-null  object
 4   PatientID             129353 non-null  int64 
 5   ProviderID            129353 non-null  int64 
 6   AppointmentDateTime   129353 non-null  object
 7   AppointmentType       129353 non-null  object
 8   Status                129353 non-null  object
 9   BookingChannel        129353 non-null  object
 10  BookingLeadTimeDays   129353 non-null  int64 
 11  IsHoliday             129353 non-null  bool  
 12  HolidayName           939 non-null     object
 13  SeasonFlag            129353 non-null  object
 14  ChronicConditionFlag  129353 non-null  bool  
d

### Inspecting data for essential cleaning checks
- Duplicates
- Null Values
- Invalid Data format
- Categorical Columns
- Leading and trailing spaces


In [30]:
print('Inspecting the dataset... ')

print('Percentage of null values in each column:')
print(df.isnull().sum()/df.shape[0]*100)
print('\nChecking for duplicate rows in the dataset...')
print(f'Number of duplicate rows: {df.duplicated().sum()}')
print('Checking for duplicated AppointmentIDs...')
print(f'Number of duplicated AppointmentIDs: {df["AppointmentID"].duplicated().sum()}')

Inspecting the dataset... 
Percentage of null values in each column:
AppointmentID            0.000000
ClinicID                 0.000000
ClinicName               0.000000
ClinicType               0.000000
PatientID                0.000000
ProviderID               0.000000
AppointmentDateTime      0.000000
AppointmentType          0.000000
Status                   0.000000
BookingChannel           0.000000
BookingLeadTimeDays      0.000000
IsHoliday                0.000000
HolidayName             99.274079
SeasonFlag               0.000000
ChronicConditionFlag     0.000000
dtype: float64

Checking for duplicate rows in the dataset...
Number of duplicate rows: 0
Checking for duplicated AppointmentIDs...
Number of duplicated AppointmentIDs: 0


In [58]:
# Validating Dates and time columns
print('Checking for invalid date formats in the dataset...')
df['AppointmentDateTime'] = pd.to_datetime(df['AppointmentDateTime'], errors='coerce')
print('Invalid dates in AppointmentDateTime column:')
print(df[df['AppointmentDateTime'].isnull()].shape[0])

Checking for invalid date formats in the dataset...
Invalid dates in AppointmentDateTime column:
0


In [49]:
# Checking for Categorical variables and their unique values
 
categorical_columns = [
    "ClinicName",
    "ClinicType",
    "AppointmentType",
    "Status",
    "BookingChannel",
    "SeasonFlag"
]
for col in categorical_columns:
    print(f'Column: {col}')
    print(f'Unique values: {df[col].unique()}')
    print(f'Number of unique values: {df[col].nunique()}')
    print(f'Value counts:\n{df[col].value_counts(dropna=False)}')
    print('---')



Column: ClinicName
Unique values: ['Nexora Care - Lakeside' 'Nexora Care - Riverside'
 'Nexora Care - Downtown Express' 'Nexora Care - Hillcrest']
Number of unique values: 4
Value counts:
Nexora Care - Lakeside            45681
Nexora Care - Riverside           29382
Nexora Care - Hillcrest           29144
Nexora Care - Downtown Express    25146
Name: ClinicName, dtype: int64
---
Column: ClinicType
Unique values: ['Multi-Specialty' 'Primary Care' 'Telehealth-Forward']
Number of unique values: 3
Value counts:
Primary Care          58526
Multi-Specialty       45681
Telehealth-Forward    25146
Name: ClinicType, dtype: int64
---
Column: AppointmentType
Unique values: ['New Patient' 'Follow-Up' 'Urgent' 'Telehealth']
Number of unique values: 4
Value counts:
Follow-Up      57859
Telehealth     30117
New Patient    21583
Urgent         19794
Name: AppointmentType, dtype: int64
---
Column: Status
Unique values: ['Completed' 'No-Show' 'Cancelled' 'Rescheduled']
Number of unique values: 4
Value 

In [51]:
    # Checking for leading/trailing spaces in categorical variables
    print(f'Checking for leading/trailing spaces in column: {col}')
    if df[col].dtype == 'object':
        if df[col].str.contains('^\s+|\s+$', regex=True).any():
            print(f'Column {col} contains leading/trailing spaces in some values.')
        else:
            print(f'Column {col} does not contain leading/trailing spaces in any values.')

Checking for leading/trailing spaces in column: SeasonFlag
Column SeasonFlag does not contain leading/trailing spaces in any values.
Column SeasonFlag does not contain leading/trailing spaces in any values.


### Inspecting Booking Lead Time and Clinic ID/Name consistency
- Booking Lead-time
- Negative Lead-time
- Clinic ID and Name consistency

In [59]:
print('Summary statistics of the BookingLeadTimeDays column:')
print(df['BookingLeadTimeDays'].describe().to_string())

Summary statistics of the BookingLeadTimeDays column:
count    129353.000000
mean          8.704267
std           7.641921
min           0.000000
25%           3.000000
50%           7.000000
75%          12.000000
max          60.000000


In [63]:
# Check negative lead times
df[df["BookingLeadTimeDays"] < 0]


# Check clinic ID and name consistency
df[["ClinicID", "ClinicName"]].drop_duplicates()
df.groupby("ClinicID")["ClinicName"].nunique()

ClinicID
1    1
2    1
3    1
4    1
Name: ClinicName, dtype: int64

### Clinic Identifier Validation

The dataset contains four unique clinics. Each clinic ID corresponds to one clinic name, and each clinic name is associated with one clinic ID. There are no missing clinic identifiers or names.

No inconsistencies were identified, so no cleaning was required for these columns.

### Validating Dates and time columns
- Valid dates
- missing dates
- date range
- future dates
- Gaps across clinics

In [71]:

# Validating Dates and time columns
df["AppointmentDateTime"] = pd.to_datetime(
    df["AppointmentDateTime"],
    errors="coerce"
)

# check missing values in the AppointmentDateTime column
print('Missing values in AppointmentDateTime column:', df['AppointmentDateTime'].isnull().sum())

#check date range of the AppointmentDateTime column
print('Date range of AppointmentDateTime column:', df['AppointmentDateTime'].agg(['min', 'max']))

# checking future dates in the AppointmentDateTime column
print('Future dates in AppointmentDateTime column:', df[df['AppointmentDateTime'] > pd.Timestamp.now()].shape[0])


Missing values in AppointmentDateTime column: 0
Date range of AppointmentDateTime column: min   2024-01-01 08:00:00
max   2025-12-31 17:45:00
Name: AppointmentDateTime, dtype: datetime64[ns]
Future dates in AppointmentDateTime column: 0


In [75]:
# Ching for gaps in date across clinics

for clinic, group in df.groupby("ClinicName"):

    dates = group["AppointmentDateTime"]

    clinic_dates = pd.DatetimeIndex(
        dates.dt.normalize().unique()
    )

    expected_dates = pd.date_range(
        dates.min().normalize(),
        dates.max().normalize(),
        freq="D"
    )

    missing = expected_dates.difference(clinic_dates)

    print(clinic.replace("Nexora Care - ", "").strip())
    print("First appointment:", dates.min())
    print("Missing dates:", len(missing))
    print(pd.Series(missing.day_name()).value_counts())
    print()

Downtown Express
First appointment: 2024-01-01 09:45:00
Missing dates: 0
Series([], dtype: int64)

Hillcrest
First appointment: 2024-06-29 08:15:00
Missing dates: 79
Sunday    79
dtype: int64

Lakeside
First appointment: 2024-01-01 08:00:00
Missing dates: 104
Sunday    104
dtype: int64

Riverside
First appointment: 2024-01-01 08:30:00
Missing dates: 104
Sunday    104
dtype: int64



### Date Validation

The appointment dates were examined to identify missing values, invalid dates, future dates, and unexpected gaps.

All 129,353 records contained valid appointment dates, covering 1 January 2024 to 31 December 2025. No future dates or missing dates were identified across the  dataset.

However, clinic-level checks revealed that Riverside, Lakeside, and Hillcrest had no appointment records on Sundays. Hillcrest's records also began on 29 June 2024, unlike the other three clinics.

These gaps may reflect differences in clinic operating schedules rather than data quality issues. They were therefore retained for further investigation.

No records were removed or modified during date validation.

### Validating Logical Consistencies
- HolidayName and isHoliday
- seasonalFlags and AppointmentDates
- appointmentStatus consistency
- BookingLeadTime

In [86]:
#check consistency between HolidayDay and isHoliday columns
holiday_inconsistencies = df.groupby(
    ["IsHoliday", "HolidayName"],
    dropna=False
).size()
print(holiday_inconsistencies)

IsHoliday  HolidayName           
False      NaN                       128414
True       Christmas Day                 80
           Christmas Eve                105
           Day After Thanksgiving        64
           Independence Day              63
           Juneteenth                    72
           Labor Day                     58
           MLK Day                       59
           Memorial Day                  58
           New Year's Day                74
           New Year's Eve                87
           Presidents' Day               73
           Thanksgiving                  87
           Veterans Day                  59
dtype: int64


In [93]:
#consistency between season flag and appointment date
season_inconsistencies = df.groupby(
    [df["AppointmentDateTime"].dt.month, "SeasonFlag"]
).size()
print(season_inconsistencies)

AppointmentDateTime  SeasonFlag    
1                    Flu Season        10939
2                    Flu Season        10341
3                    Allergy Season     9872
4                    Allergy Season    10640
5                    Allergy Season    10212
6                    Standard           8796
7                    Standard          10887
8                    Standard          11333
9                    Standard          10934
10                   Standard          11946
11                   Standard          10119
12                   Flu Season        13334
dtype: int64


In [94]:
# Appointment Status consistency check
appointment_status_inconsistencies = df['Status'].value_counts(dropna=False)
print(appointment_status_inconsistencies)

Completed      109040
No-Show         11367
Cancelled        6296
Rescheduled      2650
Name: Status, dtype: int64


In [95]:
# Check Bookingleadtime consistency
print(df['BookingLeadTimeDays'].agg([min,max]))

min     0
max    60
Name: BookingLeadTimeDays, dtype: int64


### Investigating Outliers in BookingLeadTimeDay
- use the IQR to identify outliers
- investigate daily appointment volume outlier
- Investigate outlier for each clinic


In [111]:
df['BookingLeadTimeDays'].describe() # already done earlier but repeated

# use IQR to calculate the number of outliers in the BookingLeadTimeDays column

Q1 = df['BookingLeadTimeDays'].quantile(0.25)
Q3 = df['BookingLeadTimeDays'].quantile(0.75)


IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

lead_outliers = df[
    (df["BookingLeadTimeDays"] < lower_bound) |
    (df['BookingLeadTimeDays'] > upper_bound)
]




print("Number of outliers:", len(lead_outliers))
print("Percentage:", round(len(lead_outliers) / len(df) * 100, 2))




Number of outliers: 4833
Percentage: 3.74


In [114]:
# Checking for Unusual Busy or quiet days across clinics

Q1 = daily_volume.groupby("ClinicName")["AppointmentVolume"].transform(
    lambda x: x.quantile(0.25)
)

Q3 = daily_volume.groupby("ClinicName")["AppointmentVolume"].transform(
    lambda x: x.quantile(0.75)
    )

df["ClinicName"] = df["ClinicName"].str.replace("Nexora Care - ", "", regex=False)

daily_volume = (
    df.groupby([
        "ClinicName",
        df["AppointmentDateTime"].dt.date
    ])
    .size()
    .reset_index(name="AppointmentVolume")
)

daily_volume.head()

volume_outliers = daily_volume[
    (daily_volume["AppointmentVolume"] < lower_bound) |
    (daily_volume["AppointmentVolume"] > upper_bound)
].copy()

volume_outliers = daily_volume[
    (daily_volume["AppointmentVolume"] < lower_bound) |
    (daily_volume["AppointmentVolume"] > upper_bound)
].copy()

print("Number of volume outliers:", len(volume_outliers))
print("Percentage:", round(len(volume_outliers) / len(daily_volume) * 100, 2))

volume_outliers.tail(10)

Number of volume outliers: 2055
Percentage: 83.64


,ClinicName,AppointmentDateTime,AppointmentVolume
2442,Riverside,2025-12-15,52
2443,Riverside,2025-12-16,82
2444,Riverside,2025-12-17,68
2445,Riverside,2025-12-18,61
2446,Riverside,2025-12-19,60
2448,Riverside,2025-12-22,85
2449,Riverside,2025-12-23,76
2452,Riverside,2025-12-26,87
2454,Riverside,2025-12-29,75
2455,Riverside,2025-12-30,63
